# 📊 Sales Data Analysis & Forecasting System

This notebook walks through:
1. Loading & cleaning multi-source sales data
2. Exploratory Data Analysis (EDA)
3. Feature Engineering
4. Model Training (Linear Regression, Random Forest)
5. Model Evaluation (RMSE, MAE, R²)
6. Business Dashboards

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

df = pd.read_csv('../data/sales_data.csv', parse_dates=['date'])
df.head()

## 1. Data Cleaning

In [ ]:
print('Shape before cleaning:', df.shape)
print('Missing values:\n', df.isnull().sum())

df = df.drop_duplicates()
df['temperature'] = df['temperature'].fillna(df['temperature'].mean())
df = df[df['units_sold'] >= 0]

print('Shape after cleaning:', df.shape)

## 2. Exploratory Data Analysis

In [ ]:
monthly = df.groupby(pd.Grouper(key='date', freq='ME'))['revenue'].sum().reset_index()
plt.figure(figsize=(10,5))
plt.plot(monthly['date'], monthly['revenue'], marker='o')
plt.title('Monthly Revenue Trend')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(data=df, x='category', y='units_sold', estimator=sum, errorbar=None)
plt.title('Total Units Sold by Category')
plt.show()

In [ ]:
numeric_cols = ['units_sold','price','revenue','promotion','temperature','holiday']
plt.figure(figsize=(7,6))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

## 3. Feature Engineering

In [ ]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['quarter'] = df['date'].dt.quarter
df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)

df['month_sin'] = np.sin(2*np.pi*df['month']/12)
df['month_cos'] = np.cos(2*np.pi*df['month']/12)
df['dow_sin'] = np.sin(2*np.pi*df['day_of_week']/7)
df['dow_cos'] = np.cos(2*np.pi*df['day_of_week']/7)

df = df.sort_values(['store','category','date'])
df['units_sold_lag_1'] = df.groupby(['store','category'])['units_sold'].shift(1)
df['units_sold_lag_7'] = df.groupby(['store','category'])['units_sold'].shift(7)
df['units_sold_roll_mean_7'] = df.groupby(['store','category'])['units_sold']\
    .transform(lambda x: x.shift(1).rolling(7).mean())

df = df.dropna().reset_index(drop=True)
df = pd.get_dummies(df, columns=['store','category'], drop_first=True)
df.head()

## 4. Model Training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

target = 'units_sold'
drop_cols = ['date', target, 'revenue']
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

rf = RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

## 5. Evaluation

In [ ]:
def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f'{name:20s} RMSE={rmse:8.3f}  MAE={mae:8.3f}  R2={r2:.4f}')

evaluate('Linear Regression', y_test, y_pred_lr)
evaluate('Random Forest', y_test, y_pred_rf)

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred_rf, alpha=0.3, s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Units Sold')
plt.ylabel('Predicted Units Sold')
plt.title('Random Forest: Actual vs Predicted')
plt.show()

In [ ]:
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False).head(15)
plt.figure(figsize=(8,6))
sns.barplot(x=importances.values, y=importances.index)
plt.title('Top 15 Feature Importances (Random Forest)')
plt.show()

## 6. Conclusion

- Random Forest outperforms Linear Regression across RMSE, MAE, and R² metrics.
- Key drivers of sales: lag features (recent sales history), rolling averages, promotions, and seasonality.
- The trained models and dashboards in `outputs/` can support inventory and budget planning decisions.